In [5]:
from pathlib import Path

In [7]:
from stomataMeasure.stomata_pipeline import (
    StomataPorePipeline,
    PipelineConfig,
    ScaleConfig,
    OutputConfig,
)

import stomataMeasure.stomata_pipeline as sp

In [9]:
PIPELINE_DIR = Path(sp.__file__).resolve().parent

YOLO_WEIGHTS = PIPELINE_DIR / "stomataYOLO.pt"
SAM_CKPT     = PIPELINE_DIR / "sam_vit_b_01ec64.pth"
SAM_TYPE     = "vit_b"

assert YOLO_WEIGHTS.exists()
assert SAM_CKPT.exists()

In [10]:
UM_PER_PX = (0.1 / 290.0) * 1000.0  # µm/px

pipe = StomataPorePipeline(
    yolo_weights=YOLO_WEIGHTS,
    sam_ckpt=SAM_CKPT,
    sam_type=SAM_TYPE,
    cfg=PipelineConfig(
        imgsz=2560,
        device=0,      # or "cpu" or "mps"
        conf=0.25,
        iou=0.7,
    ),
    scale=ScaleConfig(um_per_px=UM_PER_PX),
)


In [11]:
img = Path(
    "0_data/StomataImages-Annotation1-SPLIT/SPLIT1/images/test/IMG_Without_Scale11.tif"
)

out = OutputConfig(
    out_dir=Path("outputs/single"),
    save_overlay=True,
    save_excel=True,
)

res = pipe.run_image(img, out=out)

print("Overlay:", res.overlay_path)
print("Excel:", res.excel_path)
res.instances_df.head()

Overlay: outputs/single/IMG_Without_Scale11_overlay.png
Excel: outputs/single/IMG_Without_Scale11_results.xlsx


,image,uid,instance,class_id,confidence,length_px,width_px,pixel_count,length_µm,width_µm,area_µm²
0,IMG_Without_Scale11.tif,1,0,0,0.946797,158.007355,129.386185,17794,54.485295,44.615926,2115.814507
48,IMG_Without_Scale11.tif,1,45,1,0.906230,81.586754,29.725407,2107,28.133363,10.250140,250.535077
1,IMG_Without_Scale11.tif,2,1,0,0.942055,176.234390,112.365097,17281,60.770479,38.746585,2054.815696
46,IMG_Without_Scale11.tif,2,43,1,0.910039,94.051117,24.914701,2197,32.431420,8.591276,261.236623
2,IMG_Without_Scale11.tif,3,2,0,0.941470,165.381119,113.695969,16372,57.027972,39.205506,1946.730083


In [13]:
images = [
    Path("0_data/StomataImages-Annotation1-SPLIT/SPLIT1/images/test/IMG_Without_Scale11.tif"),
    Path("0_data/StomataImages-Annotation1-SPLIT/SPLIT1/images/test/IMG_Without_Scale101.tif"),
]

out = OutputConfig(
    out_dir=Path("outputs/batch_list"),
    save_overlay=True,
    save_excel=False,   # per-image Excel OFF
)

batch = pipe.run_images(
    image_paths=images,
    out=out,
    batch_excel_name="batch_results.xlsx",
    save_per_image_excel=False,
)

print("Batch Excel:", batch["batch_excel_path"])
batch["per_image_df"].head()


Batch Excel: outputs/batch_list/batch_results.xlsx


,image,H_px,W_px,µm_per_px,image_area_µm²,stomata_count_yolo,pore_count_yolo,stomata_density_per_mm²,pore_density_per_mm²,stomata_length_mean_µm,stomata_width_mean_µm,stomata_area_mean_µm²,pore_length_mean_µm,pore_width_mean_µm,pore_area_mean_µm²,raw_stomata_count_yolo,raw_pore_count_yolo,dropped_pore_count
0,IMG_Without_Scale11.tif,1920,2560,0.344828,584447.086801,37,35,63.307699,59.885661,54.877538,37.966219,1795.960407,27.452324,9.514460,236.925429,37,35,0
1,IMG_Without_Scale101.tif,1920,2560,0.344828,584447.086801,31,12,53.041585,20.532227,64.040606,36.629980,2003.095393,31.558030,8.828677,257.768530,31,12,0


In [14]:
image_folder = Path(
    "0_data/StomataImages-Annotation1-SPLIT/SPLIT1/images/test"
)

out = OutputConfig(
    out_dir=Path("outputs/batch_folder"),
    save_overlay=True,
    save_excel=False,
)

folder_batch = pipe.run_folder_batch(
    folder=image_folder,
    out=out,
    glob_pattern="*.tif",
    batch_excel_name="folder_results.xlsx",
    save_per_image_excel=False,
)

print("Batch Excel:", folder_batch["batch_excel_path"])
folder_batch["per_image_df"].head()


Batch Excel: outputs/batch_folder/folder_results.xlsx


,image,H_px,W_px,µm_per_px,image_area_µm²,stomata_count_yolo,pore_count_yolo,stomata_density_per_mm²,pore_density_per_mm²,stomata_length_mean_µm,stomata_width_mean_µm,stomata_area_mean_µm²,pore_length_mean_µm,pore_width_mean_µm,pore_area_mean_µm²,raw_stomata_count_yolo,raw_pore_count_yolo,dropped_pore_count
0,IMG_Without_Scale101.tif,1920,2560,0.344828,584447.086801,31,12,53.041585,20.532227,64.040606,36.629980,2003.095393,31.558030,8.828677,257.768530,31,12,0
1,IMG_Without_Scale11.tif,1920,2560,0.344828,584447.086801,37,35,63.307699,59.885661,54.877538,37.966219,1795.960407,27.452324,9.514460,236.925429,37,35,0
2,IMG_Without_Scale112.tif,1920,2560,0.344828,584447.086801,42,35,71.862793,59.885661,58.419135,37.027345,1839.023838,28.551898,8.694594,233.103448,42,35,0
3,IMG_Without_Scale169.tif,1920,2560,0.344828,584447.086801,19,15,32.509359,25.665283,61.245455,36.460492,1874.554102,28.369303,10.242785,266.000793,19,15,0
4,IMG_Without_Scale74.tif,1920,2560,0.344828,584447.086801,21,18,35.931396,30.798340,67.606016,30.964795,1763.631731,35.735059,9.097332,285.103713,21,19,1
